# IBM HR Employee Attrition Analysis

**Author**: JosteinCT  
**Date Created**: September 29, 2026  
**Dataset**: [Kaggle - IBM HR Analytics](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)  

---

## Executive Summary

*This project explores employee attrition patterns using exploratory data analysis (EDA) techniques in Python. The goal is to identify key factors driving employee turnover and provide actionable retention recommendations for HR leadership.*

## Business Questions

1. What factors correlate most strongly with employee attrition?
2. Which departments have highest/lowest turnover rates?
3. Is there a relationship between compensation and attrition risk?
4. How do tenure, age, and job satisfaction interact with attrition?
5. What retention strategies would you recommend based on the findings?

---

## 1. Import Libraries

*Loading required Python libraries for data manipulation, statistical analysis, and visualization.*

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Set random seed for reproducibility
np.random.seed(42)

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


---

## 2. Load & Inspect Messy Dataset

*Before any cleaning, I loaded the raw dataset with intentional data quality issues. This allows me to document the problems systematically and demonstrate my cleaning methodology.*

In [90]:
# Define file paths
MESSY_PATH = '../data/raw/IBM_HR_Employee_Attrition_MESSY.csv'
CLEAN_PATH = '../data/cleaned/IBM_HR_Employee_Attrition_CLEAN.csv'

# Load messy dataset
df_raw = pd.read_csv(MESSY_PATH)

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nFirst 5 rows:")
display(df_raw.head())

Shape: 1,520 rows × 36 columns

First 5 rows:


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,EmployeeName
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4.0,Single,NaN,19479,8,Y,yes,11,3,1,80,0,8,0,1,6,4,0,5,Employee_0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,NaN,1,2,3,Male,61,2,2,Research Scientist,2.0,Married,5130.0,24907,1,Y,NO,23,4,4,80,1,10,3,3,10,7,1,7,Employee_1
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3.0,Single,2090.0,2396,6,Y,yes,15,3,2,80,0,7,3,3,0,0,0,0,Employee_2
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3.0,Married,2909.0,23159,1,Y,yes,11,3,3,80,0,8,3,3,8,7,3,0,Employee_3
4,27.0,No,travel_rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2.0,Married,3468.0,16632,9,Y,NO,12,3,4,80,1,6,3,3,2,2,2,2,Employee_4


*Initial inspection reveals the dataset structure. Now I'll profile the data quality issues systematically.*

In [91]:
# Quick overview using built-in method
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1520 entries, 0 to 1519
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1434 non-null   float64
 1   Attrition                 1520 non-null   object 
 2   BusinessTravel            1520 non-null   object 
 3   DailyRate                 1520 non-null   int64  
 4   Department                1447 non-null   object 
 5   DistanceFromHome          1520 non-null   int64  
 6   Education                 1520 non-null   int64  
 7   EducationField            1454 non-null   object 
 8   EmployeeCount             1520 non-null   int64  
 9   EmployeeNumber            1520 non-null   int64  
 10  EnvironmentSatisfaction   1520 non-null   int64  
 11  Gender                    1520 non-null   object 
 12  HourlyRate                1520 non-null   int64  
 13  JobInvolvement            1520 non-null   int64  
 14  JobLevel

In [92]:
# Additional breakdown by data type
numeric_cols = df_raw.select_dtypes(include=['number']).columns.tolist()
cat_cols = df_raw.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical columns ({len(numeric_cols)}): {numeric_cols[:5]}...")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols[:5]}...")


Numerical columns (25): ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount']...
Categorical columns (11): ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender']...


### Missing Value Analysis

*Checking for null values across all columns. Expected: ~5% missing in Age, MonthlyIncome, JobSatisfaction, EducationField, and Department.*

In [93]:
# Check for missing values
missing_data = df_raw.isnull().sum()
missing_percentage = (missing_data / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing %': missing_percentage.round(2)
}).sort_values('Missing Count', ascending=False)

print("MISSING VALUES ANALYSIS")
print("-"*40)
print(missing_df[missing_df['Missing Count'] > 0])

MISSING VALUES ANALYSIS
----------------------------------------
                 Missing Count  Missing %
MonthlyIncome              104       6.84
Age                         86       5.66
Department                  73       4.80
JobSatisfaction             67       4.41
EducationField              66       4.34
EmployeeName                 1       0.07


### Duplicate Record Detection

*Checking for duplicate records using multiple approaches. Since EmployeeNumber should be unique per employee, any duplicates on this field indicate data quality issues.*

In [94]:
print("CHECKING FOR DUPLICATES:")
print("-"*40)

# Method 1: Check exact row duplicates (all columns)
exact_dupes = df_raw.duplicated().sum()
print(f"1. Exact duplicates (all columns): {exact_dupes:,}")

# Method 2: Check by unique identifier (EmployeeNumber)
print(f"\n2. Duplicates by EmployeeNumber (unique ID):")
duplicate_count = df_raw.duplicated(subset='EmployeeNumber').sum()
print(f"   → {duplicate_count:,} duplicate employee records found")

# Method 3: Verify against original row count (1470)
print(f"\n3. Row count verification:")
print(f"   → Current rows: {len(df_raw):,}")
print(f"   → Note: {duplicate_count:,} records flagged as duplicates need investigation")

CHECKING FOR DUPLICATES:
----------------------------------------
1. Exact duplicates (all columns): 0

2. Duplicates by EmployeeNumber (unique ID):
   → 50 duplicate employee records found

3. Row count verification:
   → Current rows: 1,520
   → Note: 50 records flagged as duplicates need investigation


---

## 3. Data Cleaning Pipeline

*Systematic cleaning of the messy dataset. Each step documents the issue found, the action taken, and the rationale.*

### Step 1: Correct Data Types

*The `info()` output revealed key data type issues:*

| Column | Current Dtype | Issue | Resolution |
|--------|---------------|-------|------------|
| `MonthlyIncome` | `object` | Contains text ('N/A', 'null', etc.) | Convert to `float64` (invalid → NaN) |
| `Age` | `float64` | Temporarily float due to NaNs | Will convert back to `int64` after imputation |
| `JobSatisfaction` | `float64` | Temporarily float due to NaNs | Will convert back to `int64` after imputation |

*Converting types first ensures accurate duplicate detection and reliable imputation statistics in subsequent steps.*

In [95]:
# Identify the issue: MonthlyIncome stored as object (text)
print("DATA TYPE CORRECTION")
print("="*40)
print(f"MonthlyIncome dtype before: {df_raw['MonthlyIncome'].dtype}")

# Show which text values are polluting the column
income_coerced = pd.to_numeric(df_raw['MonthlyIncome'], errors='coerce')
bad_income_mask = income_coerced.isna()
print(f"\nNon-numeric MonthlyIncome values found: {bad_income_mask.sum():,}")
print(f"Invalid values present: {df_raw.loc[bad_income_mask, 'MonthlyIncome'].unique()}")

# Apply the conversion: text becomes NaN (handled later in missing values step)
df_raw['MonthlyIncome'] = income_coerced
print(f"\nMonthlyIncome dtype after: {df_raw['MonthlyIncome'].dtype}")
print(f"MonthlyIncome now missing: {df_raw['MonthlyIncome'].isnull().sum():,} values (will be handled in Step 3)")

# Note on other float columns
print("\n" + "-"*40)
print("TEMPORARY FLOAT COLUMNS (due to NaN values)")
print("-"*40)
float_cols = ['Age', 'JobSatisfaction']
for col in float_cols:
    print(f"   {col}: currently float64 → will convert to int64 after imputation")

# Create working copy for the rest of the cleaning pipeline
df = df_raw.copy()
print(f"\n✓ Working copy created: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("="*40)

DATA TYPE CORRECTION
MonthlyIncome dtype before: object

Non-numeric MonthlyIncome values found: 110
Invalid values present: [nan 'unknown']

MonthlyIncome dtype after: float64
MonthlyIncome now missing: 110 values (will be handled in Step 3)

----------------------------------------
TEMPORARY FLOAT COLUMNS (due to NaN values)
----------------------------------------
   Age: currently float64 → will convert to int64 after imputation
   JobSatisfaction: currently float64 → will convert to int64 after imputation

✓ Working copy created: 1,520 rows × 36 columns


---

### Step 2: Remove Duplicate Records

*Duplicate employee records inflate counts and bias statistics. Since `EmployeeNumber` is the unique identifier per employee, any duplicates on this field indicate data quality issues.*

**Strategy:** Keep the **first occurrence** of each employee (original row), remove later duplicates.

In [96]:
# Identify duplicate employees
initial_rows = len(df)
duplicate_count = df.duplicated(subset='EmployeeNumber').sum()

print("DUPLICATE REMOVAL")
print("="*40)
print(f"Initial rows: {initial_rows:,}")
print(f"Duplicate employee records found: {duplicate_count:,}")

# Remove duplicates, keeping first occurrence
df = df.drop_duplicates(subset='EmployeeNumber', keep='first')

removed = initial_rows - len(df)
print(f"Removed: {removed:,} duplicate rows")
print(f"Final rows: {len(df):,} unique employees")

# Verify no duplicates remain
remaining_dups = df.duplicated(subset='EmployeeNumber').sum()
print(f"\nRemaining duplicates after removal: {remaining_dups:,}")
print(f"✓ All EmployeeNumbers are now unique")

DUPLICATE REMOVAL
Initial rows: 1,520
Duplicate employee records found: 50
Removed: 50 duplicate rows
Final rows: 1,470 unique employees

Remaining duplicates after removal: 0
✓ All EmployeeNumbers are now unique


---

### Step 3: Handle Missing Values & Remove Irrelevant Columns

*After duplicate removal, remaining missing values are filled using statistically appropriate methods:*

| Column Type | Imputation Strategy | Rationale |
|-------------|---------------------|-----------|
| **Numeric** (Age, MonthlyIncome, JobSatisfaction) | Fill with **median** | Less affected by outliers than mean |
| **Categorical** (EducationField, Department, etc.) | Fill with **mode** | Preserves most common category |

**Additional data reduction:** Removed `EmployeeName` column — artificial field created during data injection with no analytical value for attrition prediction.

**Note:** `Age` and `JobSatisfaction` were temporarily `float64` due to NaN values. After imputation, we convert them back to `int64` since pandas cannot store NaN in integer columns.

In [97]:
print("MISSING VALUES HANDLING & COLUMN REDUCTION")
print("="*40)

# ========== DOCUMENT CURRENT STATE ==========
print("\nCURRENT MISSING VALUE STATUS")
print("-"*40)

# Fresh count at this moment
missing_now = df.isnull().sum()
cols_with_missing = missing_now[missing_now > 0]
print(f"\nColumns with missing values: {len(cols_with_missing)}")
if len(cols_with_missing) > 0:
    print(f"\nMissing count per column:")
    for col in cols_with_missing.index:
        print(f"   {col}: {missing_now[col]:,} NaNs")

# ========== REMOVE IRRELEVANT COLUMNS ==========
print("\nIRRELEVANT COLUMNS REMOVED")
print("-"*40)

if 'EmployeeName' in df.columns:
    print(f"   Dropping 'EmployeeName' - artificial field, no analytical value")
    df = df.drop(columns=['EmployeeName'])
    print(f"   Columns remaining: {len(df.columns)}")
else:
    print(f"   EmployeeName not found (already removed)")

# ========== NUMERIC COLUMNS ==========
print("\nNUMERIC COLUMNS (median imputation)")
print("-"*40)

numeric_cols = ['Age', 'MonthlyIncome', 'JobSatisfaction']  # Focus on the ones we know have issues
for col in numeric_cols:
    nan_count = df[col].isnull().sum()
    if nan_count > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"   {col}: {nan_count} NaNs → filled with median ({median_val:.2f})")
    else:
        print(f"   {col}: No NaNs found (already clean)")

# ========== CATEGORICAL COLUMNS ==========
print("\nCATEGORICAL COLUMNS (mode imputation)")
print("-"*40)

cat_cols_to_check = ['Department', 'EducationField']
for col in cat_cols_to_check:
    if col in df.columns:
        nan_count = df[col].isnull().sum()
        if nan_count > 0:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f"   {col}: {nan_count} NaNs → filled with mode ({mode_val})")
        else:
            print(f"   {col}: No NaNs found (already clean)")

# ========== FINAL VERIFICATION ==========
print("\nFINAL MISSING VALUE CHECK")
print("-"*40)
final_missing = df.isnull().sum().sum()
print(f"Total remaining missing values: {final_missing}")

if final_missing == 0:
    print("✓ ALL MISSING VALUES HANDLED!")
else:
    print(f"⚠️ Warning: {final_missing} remaining missing values")
    print("\nRemaining columns with NaN:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

# ========== CONVERT AGE & JOBSATISFACTION TO INT64 ==========
print("\nCONVERT TEMPORARY FLOAT COLUMNS TO INTEGER")
print("-"*40)

# Verify no NaNs before converting
if 'Age' in df.columns and df['Age'].isnull().sum() == 0:
    df['Age'] = df['Age'].astype(int)
    print(f"   Age: float64 → int64 ✓")
else:
    print(f"   ⚠️ Age still has {df['Age'].isnull().sum()} NaNs - cannot convert yet")

if 'JobSatisfaction' in df.columns and df['JobSatisfaction'].isnull().sum() == 0:
    df['JobSatisfaction'] = df['JobSatisfaction'].astype(int)
    print(f"   JobSatisfaction: float64 → int64 ✓")
else:
    print(f"   ⚠️ JobSatisfaction still has {df['JobSatisfaction'].isnull().sum()} NaNs - cannot convert yet")

print(f"\n✓ Dataset ready for exploratory analysis")
print(f"   Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("="*40)

MISSING VALUES HANDLING & COLUMN REDUCTION

CURRENT MISSING VALUE STATUS
----------------------------------------

Columns with missing values: 6

Missing count per column:
   Age: 82 NaNs
   Department: 67 NaNs
   EducationField: 64 NaNs
   JobSatisfaction: 67 NaNs
   MonthlyIncome: 105 NaNs
   EmployeeName: 1 NaNs

IRRELEVANT COLUMNS REMOVED
----------------------------------------
   Dropping 'EmployeeName' - artificial field, no analytical value
   Columns remaining: 35

NUMERIC COLUMNS (median imputation)
----------------------------------------
   Age: 82 NaNs → filled with median (36.00)
   MonthlyIncome: 105 NaNs → filled with median (4898.00)
   JobSatisfaction: 67 NaNs → filled with median (3.00)

CATEGORICAL COLUMNS (mode imputation)
----------------------------------------
   Department: 67 NaNs → filled with mode (Research & Development)
   EducationField: 64 NaNs → filled with mode (Life Sciences)

FINAL MISSING VALUE CHECK
----------------------------------------
Total r

---

### Step 4: Categorical Data Validation & Standardization

*Systematic review of all categorical columns to identify:*
- **Inconsistent casing** (e.g., `'travel_frequently'` vs `'Travel_Frequently'`)
- **Typos** (e.g., `'femal'` vs `'Female'`)
- **Abbreviations** (e.g., `'hr'` vs `'Human Resources'`)
- **Special characters/encoding** (e.g., `'Björk ðæ'` in EmployeeName)

*Strategy: Document unique values first, then apply corrections systematically.*

In [98]:
print("CATEGORICAL DATA DISCOVERY")
print("="*60)

# Get all object-type columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\nCategorical columns found: {len(cat_cols)}")
print(f"Columns: {cat_cols}")

# Document unique values for each categorical column
print("\n" + "="*60)
print("UNIQUE VALUES BY COLUMN")
print("="*60)

for col in cat_cols:
    unique_vals = sorted(df[col].unique())
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(f"   {unique_vals}")

# Highlight columns with potential issues
print("\n" + "="*60)
print("POTENTIAL ISSUES FLAGGED FOR REVIEW")
print("="*60)

flagged = {}
for col in cat_cols:
    unique_lower = df[col].str.lower().str.strip().unique()
    
    # Check for mixed casing
    if len(unique_lower) != df[col].nunique():
        flagged[col] = f"Mixed casing detected ({df[col].nunique()} unique values)"
        
print(f"Columns with inconsistent casing: {list(flagged.keys())}")

print(f"\n✓ Discovery complete — ready to apply corrections")

CATEGORICAL DATA DISCOVERY

Categorical columns found: 9
Columns: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime']

UNIQUE VALUES BY COLUMN

Attrition (2 unique values):
   ['No', 'Yes']

BusinessTravel (6 unique values):
   ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely', 'non-travel', 'travel_frequently', 'travel_rarely']

Department (3 unique values):
   ['Human Resources', 'Research & Development', 'Sales']

EducationField (6 unique values):
   ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']

Gender (2 unique values):
   ['Female', 'Male']

JobRole (9 unique values):
   ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']

MaritalStatus (3 unique values):
   ['Divorced', 'Married', 'Single']

Over18 (1 unique values):
   

In [99]:
print("\nAPPLYING STANDARDIZATIONS")
print("="*60)

# ========== CORRECT CASING ==========
print("\n1. CONSISTENT TEXT CASE (Title Case)")
print("-"*40)

# Columns with identified casing issues
case_correction_cols = ['BusinessTravel', 'OverTime']

for col in case_correction_cols:
    if col in df.columns:
        original_count = df[col].nunique()
        print(f"\n   {col}: Before standardization")
        print(f"      Unique values: {sorted(df[col].unique())}")
        
        df[col] = df[col].str.title()
        
        new_count = df[col].nunique()
        print(f"   {col}: After standardization")
        print(f"      Unique values: {sorted(df[col].unique())}")

# ========== FIX OVER18 CONSTANT COLUMN ==========
print("\n2. LOW-VARIANCE COLUMNS REVIEW")
print("-"*40)

# Check for constant/near-constant columns (provide no predictive power)
over18_unique = df['Over18'].nunique()
print(f"   Over18 unique values: {over18_unique}")
print(f"   Value(s): {sorted(df['Over18'].unique())}")

if over18_unique == 1:
    print("   ⚠️  Constant column detected — removed (no analytical value)")
    df = df.drop(columns=['Over18'])
    print(f"   Columns remaining: {len(df.columns)}")
else:
    print(f"   Over18: Kept ({over18_unique} unique values)")

# ========== FINAL VERIFICATION ==========
print("\n3. POST-STANDARDIZATION VERIFICATION")
print("-"*40)

print("\nAll categorical columns after correction:")
for col in df.select_dtypes(include=['object']).columns.tolist():
    print(f"   {col}: {sorted(df[col].unique())}")

print(f"\n✓ Standardization complete")
print(f"   Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("="*60)


APPLYING STANDARDIZATIONS

1. CONSISTENT TEXT CASE (Title Case)
----------------------------------------

   BusinessTravel: Before standardization
      Unique values: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely', 'non-travel', 'travel_frequently', 'travel_rarely']
   BusinessTravel: After standardization
      Unique values: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']

   OverTime: Before standardization
      Unique values: ['NO', 'yes']
   OverTime: After standardization
      Unique values: ['No', 'Yes']

2. LOW-VARIANCE COLUMNS REVIEW
----------------------------------------
   Over18 unique values: 1
   Value(s): ['Y']
   ⚠️  Constant column detected — removed (no analytical value)
   Columns remaining: 34

3. POST-STANDARDIZATION VERIFICATION
----------------------------------------

All categorical columns after correction:
   Attrition: ['No', 'Yes']
   BusinessTravel: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
   Department: ['Human Resources', 'Re

In [100]:
print("="*60)
print("FINAL DATA QUALITY CHECKPOINT")
print("="*60)

print(f"\nFINAL DATASET STATISTICS")
print("-"*40)
print(f"   Rows: {df.shape[0]:,} (after removing 50 duplicates)")
print(f"   Columns: {df.shape[1]} (after dropping 2 irrelevant)")

print(f"\nMISSING VALUES CHECK")
print("-"*40)
print(f"   Total NaN: {df.isnull().sum().sum()}")
print(f"   Status: {'✓ All missing values handled' if df.isnull().sum().sum() == 0 else '⚠️ Issues remain'}")

print(f"\nDATA TYPES SUMMARY")
print("-"*40)
print(df.dtypes.value_counts())

print(f"\nTARGET VARIABLE DISTRIBUTION (Attrition)")
print("-"*40)
print(df['Attrition'].value_counts(normalize=True).round(2))

# Save cleaned dataset
print(f"\nSAVING CLEANED DATASET")
print("-"*40)
df.to_csv(CLEAN_PATH, index=False)
print(f"   ✓ Saved to: {CLEAN_PATH}")
print(f"   📁 {len(df):,} rows × {df.shape[1]} columns")

print("\n" + "="*60)
print("DATA CLEANING PIPELINE COMPLETE")
print("="*60)

FINAL DATA QUALITY CHECKPOINT

FINAL DATASET STATISTICS
----------------------------------------
   Rows: 1,470 (after removing 50 duplicates)
   Columns: 34 (after dropping 2 irrelevant)

MISSING VALUES CHECK
----------------------------------------
   Total NaN: 0
   Status: ✓ All missing values handled

DATA TYPES SUMMARY
----------------------------------------
int64      25
object      8
float64     1
Name: count, dtype: int64

TARGET VARIABLE DISTRIBUTION (Attrition)
----------------------------------------
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64

SAVING CLEANED DATASET
----------------------------------------
   ✓ Saved to: ../data/cleaned/IBM_HR_Employee_Attrition_CLEAN.csv
   📁 1,470 rows × 34 columns

DATA CLEANING PIPELINE COMPLETE
